# ML1 Task 2 — SVR RBF Weighted Tail Search

This notebook is a focused continuation of the SVR RBF work. The previous tail-improvement run showed that ordinary salary ranges are modeled reasonably, but RMSE is dominated by rare tails, especially very high salaries.

Main changes in this version:

1. **Sample weights** are added so the RBF SVR pays more attention to rare high-salary observations.
2. **High/low salary risk features** already embedded in target encodings are used in dense feature profiles.
3. **Alternative target transformations** are compared, especially `sqrt`, `cube-root`, and raw-scaled targets.
4. **No validation multiplier** is used.
5. **Train-only calibration** is used: none, global train ratio, and binned train ratio.
6. **Top candidates are selected only after fold-safe CV**, fixing the previous issue where the selected model had missing CV values.
7. Multiple submission files are generated for the selected and backup candidates.

The method remains within the ML1 scope: the final predictive model is still **SVR with RBF kernel**, with feature engineering, target transformation, and sample weighting.


In [1]:
# ===============================================================
# 1. Imports and global configuration
# ===============================================================
from __future__ import annotations

import json
import math
import os
import pickle
import re
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.exceptions import ConvergenceWarning
from pandas.errors import PerformanceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=PerformanceWarning)

RANDOM_STATE = 42
TRAIN_PATH = Path("train.csv")
TEST_PATH = Path("test.csv")
SAMPLE_SUBMISSION_PATH = Path("sample_submission.csv")
OUTPUT_DIR = Path("svr_tail_weighted_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "annual.pay.usd"
ID_COL = "id"

# Use FAST_MODE first. After the notebook runs end-to-end, set FAST_MODE=False for a wider search.
FAST_MODE = True

# Keep the internal-test set for final locked evaluation only.
VAL_SIZE = 0.15
INTERNAL_TEST_SIZE = 0.15
N_SPLITS_CV = 5

# Salary thresholds used for tail diagnostics and engineered risk features.
LOW_SALARY_FLOOR = 500.0
HIGH_SALARY_100K = 100_000.0
HIGH_SALARY_200K = 200_000.0
HIGH_SALARY_500K = 500_000.0
LOW_SALARY_10K = 10_000.0

# Safety clipping for final predictions only. This prevents numerical accidents, not metric gaming.
PRED_MIN_USD = 1.0
PRED_MAX_USD = 2_000_000.0

print("Configuration loaded — weighted tail search.")
print("FAST_MODE:", FAST_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


Configuration loaded — weighted tail search.
FAST_MODE: True
OUTPUT_DIR: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\svr_tail_weighted_outputs


## 2. Load raw data and create the same internal split

We keep the same 70/15/15 logic to maintain comparability with previous runs.

In [2]:
# ===============================================================
# 2. Load data and split
# ===============================================================
def normalize_text_value(value):
    """Normalize text while preserving missing values."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = value.replace("’", "'").replace("‘", "'").replace("`", "'")
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(normalize_text_value)
    # normalize common "Other" labels
    replacements = {
        "Other:": "Other",
        "Other (please specify):": "Other",
    }
    for col in ["dev.role", "industry", "first.help.source"]:
        if col in df.columns:
            df[col] = df[col].replace(replacements)
    return df


def make_region_strata(region_series: pd.Series, min_count: int = 10) -> pd.Series:
    counts = region_series.value_counts(dropna=False)
    return region_series.where(region_series.map(counts) >= min_count, "__RARE_REGION__")


def summarize_salary_split(name: str, y: pd.Series) -> Dict[str, Any]:
    y = pd.Series(y).astype(float)
    return {
        "split": name,
        "n": int(len(y)),
        "mean": float(y.mean()),
        "median": float(y.median()),
        "min": float(y.min()),
        "max": float(y.max()),
        "lt_500": int((y < LOW_SALARY_FLOOR).sum()),
        "gt_100k": int((y > HIGH_SALARY_100K).sum()),
        "gt_200k": int((y > HIGH_SALARY_200K).sum()),
        "p01": float(y.quantile(0.01)),
        "p05": float(y.quantile(0.05)),
        "p25": float(y.quantile(0.25)),
        "p75": float(y.quantile(0.75)),
        "p95": float(y.quantile(0.95)),
        "p99": float(y.quantile(0.99)),
    }


df_all = normalize_object_columns(pd.read_csv(TRAIN_PATH))
df_kaggle = normalize_object_columns(pd.read_csv(TEST_PATH))

assert TARGET in df_all.columns, f"{TARGET} missing from train.csv"
assert ID_COL in df_kaggle.columns, f"{ID_COL} missing from test.csv"

strata_full = make_region_strata(df_all["region"]) if "region" in df_all.columns else None

df_trainval, df_internal_test = train_test_split(
    df_all,
    test_size=INTERNAL_TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=strata_full,
)

strata_trainval = make_region_strata(df_trainval["region"]) if "region" in df_trainval.columns else None
relative_val_size = VAL_SIZE / (1.0 - INTERNAL_TEST_SIZE)

df_train, df_val = train_test_split(
    df_trainval,
    test_size=relative_val_size,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=strata_trainval,
)

# Reset indices so later masks align cleanly.
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_internal_test = df_internal_test.reset_index(drop=True)
df_kaggle = df_kaggle.reset_index(drop=True)

y_train_raw = df_train[TARGET].astype(float).reset_index(drop=True)
y_val_raw = df_val[TARGET].astype(float).reset_index(drop=True)
y_internal_test_raw = df_internal_test[TARGET].astype(float).reset_index(drop=True)

split_summary = pd.DataFrame([
    summarize_salary_split("train", y_train_raw),
    summarize_salary_split("validation", y_val_raw),
    summarize_salary_split("internal_test", y_internal_test_raw),
])

print("Raw shapes")
print("train:", df_all.shape)
print("kaggle:", df_kaggle.shape)
print("\nSplit sizes")
print("train:", df_train.shape)
print("validation:", df_val.shape)
print("internal_test:", df_internal_test.shape)
print("kaggle:", df_kaggle.shape)
print("\nSalary split summary")
display(split_summary)

split_summary.to_csv(OUTPUT_DIR / "split_salary_summary.csv", index=False)


Raw shapes
train: (2512, 41)
kaggle: (628, 41)

Split sizes
train: (1758, 41)
validation: (377, 41)
internal_test: (377, 41)
kaggle: (628, 41)

Salary split summary


,split,n,mean,median,min,max,lt_500,gt_100k,gt_200k,p01,p05,p25,p75,p95,p99
0,train,1758,51287.046075,41854.5,11.0,4773360.0,57,143,13,76.26,882.55,16073.5,67898.0,117887.45,178901.9
1,validation,377,46288.106101,38113.0,1.0,384552.0,13,36,3,183.80,756.20,15122.0,65008.0,121496.40,169134.0
2,internal_test,377,45797.676393,39989.0,25.0,911275.0,11,23,2,148.76,809.20,18590.0,60052.0,116545.40,156503.0


## 3. Feature-engineering configuration

This remains SVR-specific and dense: counts, target encodings, salary-score summaries, and optional small top-N technology binaries.

In [3]:
# ===============================================================
# 3. Feature engineering configuration
# ===============================================================
MULTI_SEP = ";"

NUMERIC_BASE_COLS = [
    "coding.years.total",
    "coding.years.professional",
    "experience.years",
    "job.satisfaction",
]

# Ordinal maps. These create dense numeric features.
ORDINAL_MAPS = {
    "age.group": {"18-24": 21, "25-34": 29.5, "35-44": 39.5, "45-54": 49.5, "55+": 60},
    "education": {
        "Primary/elementary school": 0,
        "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": 1,
        "Some college/university study without earning a degree": 2,
        "Associate degree (A.A., A.S., etc.)": 3,
        "Something else": 3,
        "Bachelor's degree (B.A., B.S., B.Eng., etc.)": 4,
        "Master's degree (M.A., M.S., M.Eng., MBA, etc.)": 5,
        "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": 6,
    },
    "company.size": {
        "Just me - I am a freelancer, sole proprietor, etc.": 0,
        "2 to 9 employees": 1,
        "10 to 19 employees": 2,
        "20 to 99 employees": 3,
        "100 to 499 employees": 4,
        "500 to 999 employees": 5,
        "1,000 to 4,999 employees": 6,
        "5,000 to 9,999 employees": 7,
        "10,000 or more employees": 8,
    },
    "tech.purchase.influence": {
        "I have little or no influence": 0,
        "I have some influence": 1,
        "I have a great deal of influence": 2,
    },
    "ai.sentiment": {
        "Very unfavorable": 0,
        "Unfavorable": 1,
        "Indifferent": 2,
        "Unsure": 2,
        "Favorable": 3,
        "Very favorable": 4,
    },
    "ai.trust": {
        "Highly distrust": 0,
        "Somewhat distrust": 1,
        "Neither trust nor distrust": 2,
        "Somewhat trust": 3,
        "Highly trust": 4,
    },
    "ai.complex.rating": {
        "Very poor at handling complex tasks": 0,
        "Bad at handling complex tasks": 1,
        "Neither good or bad at handling complex tasks": 2,
        "Good, but not great at handling complex tasks": 3,
        "Very well at handling complex tasks": 4,
    },
    "daily.search.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
    "daily.answer.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
}

QUASI_ORDINAL_MAPS = {
    "is.dev.professional": {
        "I am a developer by profession": 1,
        "I am not primarily a developer, but I write code sometimes as part of my work/studies": 0,
    },
    "people.manager": {"People manager": 1, "Individual contributor": 0},
    "uses.ai": {"Yes": 2, "No, but I plan to soon": 1, "No, and I don't plan to": 0},
}

# build.vs.buy is not treated as ordinal here. We target-encode it as a category.
CATEGORICAL_TE_COLS = [
    "region",
    "dev.role",
    "industry",
    "employment.type",
    "work.location",
    "company.size",
    "education",
    "cloud.hosting",
    "people.manager",
    "is.dev.professional",
    "uses.ai",
    "build.vs.buy",
    "first.help.source",
    "ai.job.threat",
]

# Interactions for target encoding. These are dense high-signal features.
INTERACTION_TE_PAIRS = [
    ("region", "dev.role"),
    ("region", "employment.type"),
    ("region", "industry"),
    ("region", "work.location"),
]

MULTI_SELECT_COLS = [
    "prog.languages",
    "databases",
    "cloud.platforms",
    "web.frameworks",
    "other.tech",
    "dev.tools",
    "dev.environments",
    "personal.os",
    "work.os",
    "project.mgmt.tools",
    "comm.tools",
    "ai.search.tools",
    "ai.tools.used",
    "side.coding",
    "how.learned.coding",
]

# Feature profiles control how many sparse technology binaries are added.
# top0_dense has zero technology binaries and relies on counts + salary-score features.
FEATURE_PROFILES = {
    "top0_dense": {"top_n_binary_per_multiselect": 0, "include_basic_ohe": False},
    "top3_dense": {"top_n_binary_per_multiselect": 3, "include_basic_ohe": False},
    "top5_dense": {"top_n_binary_per_multiselect": 5, "include_basic_ohe": False},
    "top5_hybrid_ohe": {"top_n_binary_per_multiselect": 5, "include_basic_ohe": True},
}

# Keep search manageable first.
if FAST_MODE:
    # Small but diverse set: no tech binaries vs current best-style top5 binaries.
    FEATURE_PROFILE_LIST = ["top0_dense", "top5_dense"]
else:
    FEATURE_PROFILE_LIST = list(FEATURE_PROFILES.keys())

print("Feature profiles:", FEATURE_PROFILE_LIST)


Feature profiles: ['top0_dense', 'top5_dense']


## 4. Target transformations

We add cube-root and raw-scaled targets because log targets compress the upper salary tail too strongly for USD RMSE.

In [4]:
# ===============================================================
# 4. Target transformation helper
# ===============================================================
@dataclass
class TargetTransformer:
    variant: str
    scaler_: StandardScaler = field(default_factory=StandardScaler)
    upper_clip_: Optional[float] = None
    lower_clip_: Optional[float] = None

    def _prepare_raw(self, y_raw: pd.Series, fit: bool = False) -> np.ndarray:
        y = pd.Series(y_raw).astype(float).to_numpy()
        y = np.clip(y, PRED_MIN_USD, None)

        if self.variant.endswith("floor500") or "floor500" in self.variant:
            y = np.clip(y, LOW_SALARY_FLOOR, None)

        if "upper_clip" in self.variant:
            if fit:
                self.upper_clip_ = float(np.quantile(y, 0.995))
            y = np.clip(y, None, self.upper_clip_)

        return y

    def _transform_unscaled(self, y_raw: pd.Series, fit: bool = False) -> np.ndarray:
        y = self._prepare_raw(y_raw, fit=fit)
        if self.variant.startswith("log"):
            return np.log(y)
        if self.variant.startswith("sqrt"):
            return np.sqrt(y)
        if self.variant.startswith("cbrt"):
            return np.cbrt(y)
        if self.variant.startswith("raw"):
            return y
        raise ValueError(f"Unknown target variant: {self.variant}")

    def fit(self, y_raw: pd.Series) -> "TargetTransformer":
        z = self._transform_unscaled(y_raw, fit=True).reshape(-1, 1)
        self.scaler_.fit(z)
        return self

    def transform(self, y_raw: pd.Series) -> np.ndarray:
        z = self._transform_unscaled(y_raw, fit=False).reshape(-1, 1)
        return self.scaler_.transform(z).ravel()

    def inverse_transform_to_usd(self, y_pred_scaled: np.ndarray) -> np.ndarray:
        z = self.scaler_.inverse_transform(np.asarray(y_pred_scaled).reshape(-1, 1)).ravel()
        if self.variant.startswith("log"):
            y = np.exp(z)
        elif self.variant.startswith("sqrt"):
            y = np.square(np.clip(z, 0, None))
        elif self.variant.startswith("cbrt"):
            y = np.power(z, 3)
        elif self.variant.startswith("raw"):
            y = z
        else:
            raise ValueError(f"Unknown target variant: {self.variant}")
        return np.clip(y, PRED_MIN_USD, PRED_MAX_USD)

    def inverse_transform_to_log_scale(self, y_pred_scaled: np.ndarray) -> np.ndarray:
        """Return predicted log salary. Useful for smearing calibration on log variants."""
        y_usd = self.inverse_transform_to_usd(y_pred_scaled)
        return np.log(np.clip(y_usd, PRED_MIN_USD, None))


TARGET_VARIANTS = [
    "log_raw",
    "log_floor500",
    "sqrt_raw",
    "sqrt_floor500",
    "cbrt_raw",
    "cbrt_floor500",
    "raw_scaled",
    "raw_floor500_scaled",
    "raw_upper_clip_scaled",
]
if FAST_MODE:
    # The previous run suggested sqrt/raw targets are strongest; cbrt is added to reduce log compression while remaining less extreme than raw.
    TARGET_VARIANTS = ["sqrt_raw", "cbrt_raw", "raw_scaled", "log_floor500"]

print("Target variants:", TARGET_VARIANTS)


Target variants: ['sqrt_raw', 'cbrt_raw', 'raw_scaled', 'log_floor500']


## 5. Tail-focused SVR preprocessor

The preprocessor creates dense signals: target encodings, high/low salary rates, technology salary-score features, and tail-oriented numeric interactions.

In [5]:
# ===============================================================
# 5. Dense SVR preprocessor
# ===============================================================
def parse_multiselect_cell(value, sep: str = MULTI_SEP) -> List[str]:
    if pd.isna(value):
        return []
    value = normalize_text_value(value)
    if value == "":
        return []
    return [normalize_text_value(item) for item in str(value).split(sep) if normalize_text_value(item) != ""]


def sanitize_feature_name(name: str) -> str:
    name = normalize_text_value(name)
    name = str(name).lower()
    replacements = {
        " ": "_", "/": "_", ".": "_", "(": "", ")": "", ",": "",
        "'": "", "-": "_", "+": "plus", "#": "sharp", ":": "", "!": "",
        "&": "and", "[": "", "]": "",
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    name = re.sub(r"[^a-z0-9_]+", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name or "unknown"


def exp_bucket_from_prof_years(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    return pd.cut(
        x,
        bins=[-np.inf, 1, 3, 5, 10, 20, np.inf],
        labels=["0_1", "1_3", "3_5", "5_10", "10_20", "20_plus"],
    ).astype(str).replace("nan", "__Missing__")


@dataclass
class SmoothedStats:
    mean_map: Dict[str, float]
    high100_map: Dict[str, float]
    high200_map: Dict[str, float]
    low500_map: Dict[str, float]
    count_map: Dict[str, int]
    global_mean: float
    global_high100: float
    global_high200: float
    global_low500: float


@dataclass
class TailFocusedSVRPreprocessor:
    feature_profile: str = "top5_dense"
    te_smoothing: float = 30.0
    rate_smoothing: float = 50.0
    item_smoothing: float = 20.0
    min_missing_rate: float = 0.10

    numeric_medians_: Dict[str, float] = field(default_factory=dict)
    ordinal_fallbacks_: Dict[str, float] = field(default_factory=dict)
    missing_indicator_cols_: List[str] = field(default_factory=list)
    cat_stats_: Dict[str, SmoothedStats] = field(default_factory=dict)
    interaction_stats_: Dict[str, SmoothedStats] = field(default_factory=dict)
    multi_item_stats_: Dict[str, SmoothedStats] = field(default_factory=dict)
    multi_top_items_: Dict[str, List[str]] = field(default_factory=dict)
    feature_cols_: List[str] = field(default_factory=list)
    zero_variance_cols_: List[str] = field(default_factory=list)
    global_log_mean_: float = 0.0
    global_high100_: float = 0.0
    global_high200_: float = 0.0
    global_low500_: float = 0.0

    def fit(self, X_raw: pd.DataFrame, y_raw: pd.Series) -> "TailFocusedSVRPreprocessor":
        X = normalize_object_columns(X_raw.copy())
        X = X.drop(columns=[TARGET], errors="ignore")
        y = pd.Series(y_raw).astype(float).reset_index(drop=True)
        X = X.reset_index(drop=True)

        y_log = np.log(np.clip(y, PRED_MIN_USD, None))
        self.global_log_mean_ = float(y_log.mean())
        self.global_high100_ = float((y > HIGH_SALARY_100K).mean())
        self.global_high200_ = float((y > HIGH_SALARY_200K).mean())
        self.global_low500_ = float((y < LOW_SALARY_FLOOR).mean())

        missing_rates = X.isna().mean()
        self.missing_indicator_cols_ = [c for c, r in missing_rates.items() if r >= self.min_missing_rate and r > 0]

        # Numeric and ordinal medians.
        X_num = self._make_numeric_features(X, fit_mode=True)
        self.numeric_medians_ = {
            col: float(X_num[col].median()) if X_num[col].notna().any() else 0.0
            for col in X_num.columns
        }
        for col, mapping in {**ORDINAL_MAPS, **QUASI_ORDINAL_MAPS}.items():
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                self.ordinal_fallbacks_[col] = float(mapped.median()) if mapped.notna().any() else 0.0

        # Target encoding stats for categorical variables.
        self.cat_stats_ = {}
        for col in CATEGORICAL_TE_COLS:
            if col in X.columns:
                values = self._clean_category_series(X[col])
                self.cat_stats_[col] = self._fit_smoothed_stats(values, y)

        # Add experience bucket as a target-encoded pseudo-category.
        if "coding.years.professional" in X.columns:
            exp_bucket = exp_bucket_from_prof_years(X["coding.years.professional"])
            self.cat_stats_["professional_years_bucket"] = self._fit_smoothed_stats(exp_bucket, y)

        # Interaction target encodings.
        self.interaction_stats_ = {}
        for col_a, col_b in INTERACTION_TE_PAIRS:
            if col_a in X.columns and col_b in X.columns:
                key = f"{col_a}__x__{col_b}"
                values = self._interaction_series(X[col_a], X[col_b])
                self.interaction_stats_[key] = self._fit_smoothed_stats(values, y)
        if "region" in X.columns and "coding.years.professional" in X.columns:
            key = "region__x__professional_years_bucket"
            values = self._interaction_series(X["region"], exp_bucket_from_prof_years(X["coding.years.professional"]))
            self.interaction_stats_[key] = self._fit_smoothed_stats(values, y)

        # Multi-select item stats and top items.
        self.multi_item_stats_ = {}
        self.multi_top_items_ = {}
        for col in MULTI_SELECT_COLS:
            if col not in X.columns:
                continue
            parsed = [parse_multiselect_cell(v) for v in X[col]]
            item_rows = []
            for i, items in enumerate(parsed):
                for item in items:
                    item_rows.append((item, y.iloc[i]))
            if item_rows:
                item_df = pd.DataFrame(item_rows, columns=["item", "salary"])
                self.multi_item_stats_[col] = self._fit_smoothed_stats(item_df["item"], item_df["salary"])
                counts = Counter([item for items in parsed for item in items])
                self.multi_top_items_[col] = [item for item, _ in counts.most_common(10)]
            else:
                self.multi_item_stats_[col] = self._empty_stats()
                self.multi_top_items_[col] = []

        # Determine final columns on training data.
        X_tmp = self.transform(X_raw, fit_call=True)
        self.zero_variance_cols_ = X_tmp.columns[X_tmp.nunique(dropna=False) <= 1].tolist()
        X_tmp = X_tmp.drop(columns=self.zero_variance_cols_, errors="ignore")
        self.feature_cols_ = X_tmp.columns.tolist()
        return self

    def transform(self, X_raw: pd.DataFrame, fit_call: bool = False) -> pd.DataFrame:
        X = normalize_object_columns(X_raw.copy())
        X = X.drop(columns=[TARGET], errors="ignore")
        X = X.reset_index(drop=True)
        pieces = []

        # Missing flags.
        miss = pd.DataFrame(index=X.index)
        for col in self.missing_indicator_cols_:
            if col in X.columns:
                miss[f"{col}_missing"] = X[col].isna().astype(float)
        if "company.size" in X.columns:
            miss["company_size_dontknow"] = (X["company.size"] == "I don't know").astype(float)
        if not miss.empty:
            pieces.append(miss)

        # Dense numeric features.
        X_num = self._make_numeric_features(X, fit_mode=False)
        for col, med in self.numeric_medians_.items():
            if col not in X_num.columns:
                X_num[col] = med
            else:
                X_num[col] = X_num[col].fillna(med)
        pieces.append(X_num.astype(float))

        # Target encodings for categories.
        te_df = pd.DataFrame(index=X.index)
        for col, stats in self.cat_stats_.items():
            if col == "professional_years_bucket":
                if "coding.years.professional" in X.columns:
                    values = exp_bucket_from_prof_years(X["coding.years.professional"])
                else:
                    values = pd.Series("__Missing__", index=X.index)
            else:
                values = self._clean_category_series(X[col]) if col in X.columns else pd.Series("__Missing__", index=X.index)
            safe_col = sanitize_feature_name(col)
            self._apply_stats_to_frame(te_df, safe_col, values, stats)

        # Target encodings for interactions.
        for key, stats in self.interaction_stats_.items():
            if key == "region__x__professional_years_bucket":
                if "region" in X.columns and "coding.years.professional" in X.columns:
                    values = self._interaction_series(X["region"], exp_bucket_from_prof_years(X["coding.years.professional"]))
                else:
                    values = pd.Series("__Missing__", index=X.index)
            else:
                parts = key.split("__x__")
                col_a, col_b = parts[0], parts[1]
                if col_a in X.columns and col_b in X.columns:
                    values = self._interaction_series(X[col_a], X[col_b])
                else:
                    values = pd.Series("__Missing__", index=X.index)
            safe_key = sanitize_feature_name(key)
            self._apply_stats_to_frame(te_df, safe_key, values, stats)

        if not te_df.empty:
            pieces.append(te_df.astype(float))

        # Multi-select salary score features and optional top-N binaries.
        multi_df = pd.DataFrame(index=X.index)
        top_n = int(FEATURE_PROFILES[self.feature_profile]["top_n_binary_per_multiselect"])
        for col in MULTI_SELECT_COLS:
            if col not in X.columns or col not in self.multi_item_stats_:
                continue
            parsed = [parse_multiselect_cell(v) for v in X[col]]
            prefix = sanitize_feature_name(col)
            stats = self.multi_item_stats_[col]
            counts = np.array([len(items) for items in parsed], dtype=float)
            multi_df[f"{prefix}_count"] = counts

            # Salary score summaries.
            mean_scores, max_scores, high100_mean, high100_max, high200_mean, low500_mean, low500_max = [], [], [], [], [], [], []
            for items in parsed:
                if len(items) == 0:
                    item_mean = [stats.global_mean]
                    item_high100 = [stats.global_high100]
                    item_high200 = [stats.global_high200]
                    item_low500 = [stats.global_low500]
                else:
                    item_mean = [stats.mean_map.get(item, stats.global_mean) for item in items]
                    item_high100 = [stats.high100_map.get(item, stats.global_high100) for item in items]
                    item_high200 = [stats.high200_map.get(item, stats.global_high200) for item in items]
                    item_low500 = [stats.low500_map.get(item, stats.global_low500) for item in items]
                mean_scores.append(float(np.mean(item_mean)))
                max_scores.append(float(np.max(item_mean)))
                high100_mean.append(float(np.mean(item_high100)))
                high100_max.append(float(np.max(item_high100)))
                high200_mean.append(float(np.mean(item_high200)))
                low500_mean.append(float(np.mean(item_low500)))
                low500_max.append(float(np.max(item_low500)))
            multi_df[f"{prefix}_salary_score_mean"] = mean_scores
            multi_df[f"{prefix}_salary_score_max"] = max_scores
            multi_df[f"{prefix}_high100_score_mean"] = high100_mean
            multi_df[f"{prefix}_high100_score_max"] = high100_max
            multi_df[f"{prefix}_high200_score_mean"] = high200_mean
            multi_df[f"{prefix}_low500_score_mean"] = low500_mean
            multi_df[f"{prefix}_low500_score_max"] = low500_max

            # Optional sparse top-N binaries. Keep small N only.
            if top_n > 0:
                for item in self.multi_top_items_.get(col, [])[:top_n]:
                    item_col = f"{prefix}__{sanitize_feature_name(item)}"
                    multi_df[item_col] = [float(item in items) for items in parsed]

        if not multi_df.empty:
            pieces.append(multi_df.astype(float))

        # Optional basic one-hot for very small categories only.
        if FEATURE_PROFILES[self.feature_profile].get("include_basic_ohe", False):
            small_ohe_cols = ["employment.type", "work.location", "people.manager"]
            ohe_src = pd.DataFrame(index=X.index)
            for col in small_ohe_cols:
                if col in X.columns:
                    ohe_src[col] = X[col].fillna("__Missing__").astype(str)
            if not ohe_src.empty:
                pieces.append(pd.get_dummies(ohe_src, prefix_sep="__", drop_first=True, dtype=float))

        X_out = pd.concat(pieces, axis=1) if pieces else pd.DataFrame(index=X.index)
        X_out = X_out.loc[:, ~X_out.columns.duplicated()].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)

        if fit_call:
            return X_out

        X_out = X_out.drop(columns=self.zero_variance_cols_, errors="ignore")
        # Align to training columns.
        for col in self.feature_cols_:
            if col not in X_out.columns:
                X_out[col] = 0.0
        extra_cols = [c for c in X_out.columns if c not in self.feature_cols_]
        if extra_cols:
            X_out = X_out.drop(columns=extra_cols)
        return X_out[self.feature_cols_].astype(float)

    def _make_numeric_features(self, X: pd.DataFrame, fit_mode: bool = False) -> pd.DataFrame:
        out = pd.DataFrame(index=X.index)
        # Basic numeric.
        for col in NUMERIC_BASE_COLS:
            if col in X.columns:
                out[sanitize_feature_name(col)] = pd.to_numeric(X[col], errors="coerce")
            else:
                out[sanitize_feature_name(col)] = np.nan

        total = pd.to_numeric(X.get("coding.years.total", pd.Series(np.nan, index=X.index)), errors="coerce")
        prof = pd.to_numeric(X.get("coding.years.professional", pd.Series(np.nan, index=X.index)), errors="coerce")
        exp = pd.to_numeric(X.get("experience.years", pd.Series(np.nan, index=X.index)), errors="coerce")

        out["years_before_professional"] = (total - prof).clip(lower=0)

        # Ordinal / quasi-ordinal.
        for col, mapping in {**ORDINAL_MAPS, **QUASI_ORDINAL_MAPS}.items():
            safe = sanitize_feature_name(col)
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                fallback = self.ordinal_fallbacks_.get(col, np.nan)
                out[safe] = mapped.fillna(fallback)
            else:
                out[safe] = self.ordinal_fallbacks_.get(col, 0.0)

        # Multi-select counts.
        count_cols = {}
        for col in MULTI_SELECT_COLS:
            if col in X.columns:
                counts = pd.Series([len(parse_multiselect_cell(v)) for v in X[col]], index=X.index).astype(float)
            else:
                counts = pd.Series(0.0, index=X.index)
            c_name = f"{sanitize_feature_name(col)}_raw_count"
            count_cols[col] = counts
            out[c_name] = counts
        total_tech_count = sum(count_cols.values()) if count_cols else pd.Series(0.0, index=X.index)
        out["total_multiselect_count"] = total_tech_count

        # Ratios and dense interactions. Add 1 to denominators for stability.
        out["experience_to_professional_ratio"] = (exp / (prof + 1)).clip(lower=0, upper=10)
        age_mid = out.get("age_group", pd.Series(np.nan, index=X.index))
        out["professional_years_to_age_ratio"] = (prof / age_mid.replace(0, np.nan)).clip(lower=0, upper=1)
        out["years_before_professional_ratio"] = (out["years_before_professional"] / (total + 1)).clip(lower=0, upper=1)

        lang_count = count_cols.get("prog.languages", pd.Series(0.0, index=X.index))
        cloud_count = count_cols.get("cloud.platforms", pd.Series(0.0, index=X.index))
        db_count = count_cols.get("databases", pd.Series(0.0, index=X.index))
        ai_count = count_cols.get("ai.tools.used", pd.Series(0.0, index=X.index))
        out["language_count_x_professional_years"] = lang_count * prof.fillna(0)
        out["cloud_count_x_professional_years"] = cloud_count * prof.fillna(0)
        out["database_count_x_professional_years"] = db_count * prof.fillna(0)
        out["ai_count_x_professional_years"] = ai_count * prof.fillna(0)
        out["total_tech_count_x_professional_years"] = total_tech_count * prof.fillna(0)

        manager = out.get("people_manager", pd.Series(0.0, index=X.index)).fillna(0)
        company = out.get("company_size", pd.Series(0.0, index=X.index)).fillna(0)
        influence = out.get("tech_purchase_influence", pd.Series(0.0, index=X.index)).fillna(0)
        out["manager_x_company_size"] = manager * company
        out["influence_x_company_size"] = influence * company
        out["large_company_flag"] = (company >= 6).astype(float)
        out["senior_professional_10y"] = (prof >= 10).astype(float)
        out["senior_professional_20y"] = (prof >= 20).astype(float)
        out["high_experience_10y"] = (exp >= 10).astype(float)
        out["high_experience_20y"] = (exp >= 20).astype(float)

        if "employment.type" in X.columns:
            emp = X["employment.type"].fillna("__Missing__").astype(str)
            out["is_full_time"] = (emp == "Full-time").astype(float)
            out["is_freelance"] = emp.str.contains("Freelance", case=False, na=False).astype(float)
            out["is_student"] = emp.str.contains("Student", case=False, na=False).astype(float)
            out["is_part_time"] = emp.str.contains("Part-time", case=False, na=False).astype(float)
        if "work.location" in X.columns:
            wl = X["work.location"].fillna("__Missing__").astype(str)
            out["is_remote"] = (wl == "Remote").astype(float)
            out["is_hybrid"] = wl.str.contains("Hybrid", case=False, na=False).astype(float)

        # Tail-oriented dense interactions. These are still derived only from available predictors.
        out["senior_x_large_company"] = out["senior_professional_10y"] * out["large_company_flag"]
        out["senior20_x_large_company"] = out["senior_professional_20y"] * out["large_company_flag"]
        out["manager_x_senior"] = manager * out["senior_professional_10y"]
        out["manager_x_senior20"] = manager * out["senior_professional_20y"]
        out["remote_x_senior"] = out.get("is_remote", pd.Series(0.0, index=X.index)) * out["senior_professional_10y"]
        out["fulltime_x_senior"] = out.get("is_full_time", pd.Series(0.0, index=X.index)) * out["senior_professional_10y"]
        out["freelance_x_low_experience"] = out.get("is_freelance", pd.Series(0.0, index=X.index)) * (prof.fillna(0) <= 1).astype(float)
        out["student_or_parttime_low_salary_risk"] = (out.get("is_student", pd.Series(0.0, index=X.index)) + out.get("is_part_time", pd.Series(0.0, index=X.index))).clip(upper=1)

        return out

    def _clean_category_series(self, s: pd.Series) -> pd.Series:
        return s.fillna("__Missing__").astype(str).map(normalize_text_value).fillna("__Missing__")

    def _interaction_series(self, a: pd.Series, b: pd.Series) -> pd.Series:
        a = self._clean_category_series(a)
        b = self._clean_category_series(b)
        return a.astype(str) + "__x__" + b.astype(str)

    def _fit_smoothed_stats(self, values: pd.Series, y_raw: pd.Series) -> SmoothedStats:
        values = pd.Series(values).fillna("__Missing__").astype(str).reset_index(drop=True)
        y = pd.Series(y_raw).astype(float).reset_index(drop=True)
        y_log = np.log(np.clip(y, PRED_MIN_USD, None))
        df = pd.DataFrame({"value": values, "y": y, "log_y": y_log})
        grouped = df.groupby("value")
        counts = grouped.size()
        mean_log = grouped["log_y"].mean()
        high100 = grouped["y"].apply(lambda x: float((x > HIGH_SALARY_100K).mean()))
        high200 = grouped["y"].apply(lambda x: float((x > HIGH_SALARY_200K).mean()))
        low500 = grouped["y"].apply(lambda x: float((x < LOW_SALARY_FLOOR).mean()))

        global_mean = float(y_log.mean())
        global_high100 = float((y > HIGH_SALARY_100K).mean())
        global_high200 = float((y > HIGH_SALARY_200K).mean())
        global_low500 = float((y < LOW_SALARY_FLOOR).mean())

        mean_smooth = (mean_log * counts + global_mean * self.te_smoothing) / (counts + self.te_smoothing)
        high100_smooth = (high100 * counts + global_high100 * self.rate_smoothing) / (counts + self.rate_smoothing)
        high200_smooth = (high200 * counts + global_high200 * self.rate_smoothing) / (counts + self.rate_smoothing)
        low500_smooth = (low500 * counts + global_low500 * self.rate_smoothing) / (counts + self.rate_smoothing)

        return SmoothedStats(
            mean_map=mean_smooth.to_dict(),
            high100_map=high100_smooth.to_dict(),
            high200_map=high200_smooth.to_dict(),
            low500_map=low500_smooth.to_dict(),
            count_map=counts.astype(int).to_dict(),
            global_mean=global_mean,
            global_high100=global_high100,
            global_high200=global_high200,
            global_low500=global_low500,
        )

    def _empty_stats(self) -> SmoothedStats:
        return SmoothedStats({}, {}, {}, {}, {}, self.global_log_mean_, self.global_high100_, self.global_high200_, self.global_low500_)

    def _apply_stats_to_frame(self, out: pd.DataFrame, prefix: str, values: pd.Series, stats: SmoothedStats) -> None:
        values = pd.Series(values).fillna("__Missing__").astype(str)
        out[f"{prefix}_te_log_mean"] = values.map(stats.mean_map).fillna(stats.global_mean).astype(float)
        out[f"{prefix}_te_high100_rate"] = values.map(stats.high100_map).fillna(stats.global_high100).astype(float)
        out[f"{prefix}_te_high200_rate"] = values.map(stats.high200_map).fillna(stats.global_high200).astype(float)
        out[f"{prefix}_te_low500_rate"] = values.map(stats.low500_map).fillna(stats.global_low500).astype(float)
        out[f"{prefix}_te_log_count"] = np.log1p(values.map(stats.count_map).fillna(0).astype(float))

print("TailFocusedSVRPreprocessor ready.")


TailFocusedSVRPreprocessor ready.


## 6. Metrics, train-only calibration, and sample weights

This version removes validation multipliers. Calibration is based on training residuals only. Sample weights are added to make SVR care more about rare high salaries.

In [6]:
# ===============================================================
# 6. Metrics, train-only calibration, and sample weights
# ===============================================================
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def salary_bucket_summary(y_true: pd.Series, y_pred: np.ndarray) -> pd.DataFrame:
    df = pd.DataFrame({"actual": np.asarray(y_true, dtype=float), "pred": np.asarray(y_pred, dtype=float)})
    df["error"] = df["pred"] - df["actual"]
    df["sq_error"] = df["error"] ** 2
    df["abs_error"] = df["error"].abs()
    df["bucket"] = pd.cut(
        df["actual"],
        bins=[0, 500, 10_000, 30_000, 60_000, 100_000, 200_000, np.inf],
        labels=["<500", "500-10k", "10k-30k", "30k-60k", "60k-100k", "100k-200k", "200k+"],
        include_lowest=True,
    )
    out = df.groupby("bucket", observed=False).agg(
        n=("actual", "size"),
        mean_actual=("actual", "mean"),
        mean_pred=("pred", "mean"),
        rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
        mae=("abs_error", "mean"),
        mse_sum=("sq_error", "sum"),
    ).reset_index()
    out["mse_contribution_pct"] = 100 * out["mse_sum"] / df["sq_error"].sum()
    return out


# -------------------------------
# Sample-weight schemes
# -------------------------------
WEIGHT_SCHEMES = {
    "none": {"lt10k": 1.0, "gt100k": 1.0, "gt200k": 1.0, "gt500k": 1.0},
    "high_mild": {"lt10k": 1.0, "gt100k": 2.0, "gt200k": 4.0, "gt500k": 6.0},
    "high_medium": {"lt10k": 1.0, "gt100k": 3.0, "gt200k": 7.0, "gt500k": 12.0},
    "tail_balanced": {"lt10k": 2.0, "gt100k": 3.0, "gt200k": 7.0, "gt500k": 12.0},
    "tail_strong": {"lt10k": 2.5, "gt100k": 4.0, "gt200k": 10.0, "gt500k": 18.0},
}


def make_sample_weights(y_raw: pd.Series, scheme: str) -> np.ndarray:
    y = pd.Series(y_raw).astype(float).to_numpy()
    cfg = WEIGHT_SCHEMES.get(scheme)
    if cfg is None:
        raise ValueError(f"Unknown weight scheme: {scheme}")
    w = np.ones(len(y), dtype=float)
    w[y < LOW_SALARY_10K] = cfg["lt10k"]
    w[y > HIGH_SALARY_100K] = cfg["gt100k"]
    w[y > HIGH_SALARY_200K] = cfg["gt200k"]
    w[y > HIGH_SALARY_500K] = cfg["gt500k"]
    # Normalize mean weight to 1 so C remains comparable across schemes.
    w = w / np.mean(w)
    return w


def sample_weight_summary(y_raw: pd.Series, scheme: str) -> Dict[str, float]:
    w = make_sample_weights(y_raw, scheme)
    return {
        "weight_scheme": scheme,
        "weight_mean": float(np.mean(w)),
        "weight_min": float(np.min(w)),
        "weight_max": float(np.max(w)),
        "n_weight_gt1": int((w > 1.01).sum()),
    }


# -------------------------------
# Train-only calibration
# -------------------------------
def fit_train_global_ratio(pred_train_usd: np.ndarray, y_train_raw: pd.Series, cap_low: float = 0.65, cap_high: float = 2.25) -> float:
    pred = np.clip(np.asarray(pred_train_usd, dtype=float), PRED_MIN_USD, PRED_MAX_USD)
    true = np.clip(pd.Series(y_train_raw).astype(float).to_numpy(), PRED_MIN_USD, PRED_MAX_USD)
    ratios = np.clip(true / pred, cap_low, cap_high)
    return float(np.mean(ratios))


@dataclass
class BinnedRatioCalibrator:
    n_bins: int = 6
    min_bin_size: int = 25
    smoothing: float = 40.0
    cap_low: float = 0.60
    cap_high: float = 2.75
    global_factor_: float = 1.0
    bin_edges_: Optional[np.ndarray] = None
    bin_factors_: Optional[np.ndarray] = None

    def fit(self, pred_train_usd: np.ndarray, y_train_raw: pd.Series) -> "BinnedRatioCalibrator":
        pred = np.clip(np.asarray(pred_train_usd, dtype=float), PRED_MIN_USD, PRED_MAX_USD)
        true = np.clip(pd.Series(y_train_raw).astype(float).to_numpy(), PRED_MIN_USD, PRED_MAX_USD)
        pred_log = np.log(pred)
        ratios = np.clip(true / pred, self.cap_low, self.cap_high)
        self.global_factor_ = float(np.mean(ratios))
        edges = np.unique(np.quantile(pred_log, np.linspace(0, 1, self.n_bins + 1)))
        if len(edges) <= 2:
            self.bin_edges_ = np.array([-np.inf, np.inf])
            self.bin_factors_ = np.array([self.global_factor_])
            return self
        edges[0] = -np.inf
        edges[-1] = np.inf
        self.bin_edges_ = edges
        bin_ids = np.digitize(pred_log, edges[1:-1], right=True)
        factors = []
        for b in range(len(edges) - 1):
            mask = bin_ids == b
            n = int(mask.sum())
            if n < self.min_bin_size:
                factors.append(self.global_factor_)
            else:
                raw_factor = float(np.mean(ratios[mask]))
                smoothed = (raw_factor * n + self.global_factor_ * self.smoothing) / (n + self.smoothing)
                factors.append(float(np.clip(smoothed, self.cap_low, self.cap_high)))
        self.bin_factors_ = np.array(factors)
        return self

    def apply(self, pred_usd: np.ndarray) -> np.ndarray:
        pred = np.clip(np.asarray(pred_usd, dtype=float), PRED_MIN_USD, PRED_MAX_USD)
        if self.bin_edges_ is None or self.bin_factors_ is None:
            return np.clip(pred * self.global_factor_, PRED_MIN_USD, PRED_MAX_USD)
        pred_log = np.log(pred)
        bin_ids = np.digitize(pred_log, self.bin_edges_[1:-1], right=True)
        factors = self.bin_factors_[bin_ids]
        return np.clip(pred * factors, PRED_MIN_USD, PRED_MAX_USD)


def evaluate_calibrations(
    target_transformer: TargetTransformer,
    pred_train_scaled: np.ndarray,
    pred_val_scaled: np.ndarray,
    y_train_raw_used: pd.Series,
    y_val_raw_eval: pd.Series,
) -> Dict[str, Any]:
    pred_train_usd = target_transformer.inverse_transform_to_usd(pred_train_scaled)
    pred_val_usd_none = target_transformer.inverse_transform_to_usd(pred_val_scaled)

    records = []
    records.append({
        "calibration": "none",
        "factor_or_info": "direct",
        "pred_val_usd": pred_val_usd_none,
        "val_rmse": rmse(y_val_raw_eval, pred_val_usd_none),
    })

    # Global train-only ratio calibration. Unlike validation_multiplier, this is fitted only on training residuals.
    global_ratio = fit_train_global_ratio(pred_train_usd, y_train_raw_used)
    pred_global = np.clip(pred_val_usd_none * global_ratio, PRED_MIN_USD, PRED_MAX_USD)
    records.append({
        "calibration": "train_global_ratio",
        "factor_or_info": global_ratio,
        "pred_val_usd": pred_global,
        "val_rmse": rmse(y_val_raw_eval, pred_global),
    })

    # Binned train-only ratio calibration, designed to correct underprediction differently across prediction ranges.
    binned = BinnedRatioCalibrator(n_bins=6).fit(pred_train_usd, y_train_raw_used)
    pred_binned = binned.apply(pred_val_usd_none)
    records.append({
        "calibration": "train_binned_ratio",
        "factor_or_info": {
            "global_factor": binned.global_factor_,
            "bin_factors": binned.bin_factors_.tolist() if binned.bin_factors_ is not None else None,
        },
        "pred_val_usd": pred_binned,
        "val_rmse": rmse(y_val_raw_eval, pred_binned),
        "calibrator": binned,
    })

    best = min(records, key=lambda r: r["val_rmse"])
    return {"records": records, "best": best}

print("Metrics, sample weights, and train-only calibration helpers ready.")


Metrics, sample weights, and train-only calibration helpers ready.


## 7. Candidate evaluation

Every candidate is defined by feature profile, target transformation, RBF SVR hyperparameters, and sample-weight scheme.

In [7]:
# ===============================================================
# 7. Candidate fitting and evaluation with sample weights
# ===============================================================
def prepare_candidate_data(
    feature_profile: str,
    target_variant: str,
    drop_low_salary_rows: bool,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    kaggle_df: Optional[pd.DataFrame] = None,
):
    train_mask = pd.Series(True, index=train_df.index)
    if drop_low_salary_rows:
        train_mask = train_df[TARGET].astype(float) >= LOW_SALARY_FLOOR

    train_fit = train_df.loc[train_mask].reset_index(drop=True)
    y_train_fit_raw = train_fit[TARGET].astype(float).reset_index(drop=True)

    pre = TailFocusedSVRPreprocessor(feature_profile=feature_profile)
    pre.fit(train_fit.drop(columns=[TARGET]), y_train_fit_raw)

    X_train = pre.transform(train_fit.drop(columns=[TARGET]))
    X_val = pre.transform(val_df.drop(columns=[TARGET]))
    X_kaggle = pre.transform(kaggle_df) if kaggle_df is not None else None

    scaler_x = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler_x.fit_transform(X_train), columns=X_train.columns)
    X_val_scaled = pd.DataFrame(scaler_x.transform(X_val), columns=X_train.columns)
    X_kaggle_scaled = pd.DataFrame(scaler_x.transform(X_kaggle), columns=X_train.columns) if X_kaggle is not None else None

    target_tf = TargetTransformer(target_variant).fit(y_train_fit_raw)
    y_train_scaled = pd.Series(target_tf.transform(y_train_fit_raw))

    return {
        "preprocessor": pre,
        "scaler_x": scaler_x,
        "target_tf": target_tf,
        "X_train": X_train_scaled,
        "X_val": X_val_scaled,
        "X_kaggle": X_kaggle_scaled,
        "y_train_scaled": y_train_scaled,
        "y_train_raw_used": y_train_fit_raw,
        "n_train_rows": len(train_fit),
        "n_features": X_train_scaled.shape[1],
        "train_mask": train_mask,
    }


def evaluate_candidate(
    feature_profile: str,
    target_variant: str,
    svr_params: Dict[str, Any],
    weight_scheme: str,
    drop_low_salary_rows: bool = False,
    verbose: bool = False,
) -> Dict[str, Any]:
    start = time.time()
    prepared = prepare_candidate_data(
        feature_profile=feature_profile,
        target_variant=target_variant,
        drop_low_salary_rows=drop_low_salary_rows,
        train_df=df_train,
        val_df=df_val,
        kaggle_df=None,
    )
    model = SVR(kernel="rbf", cache_size=1000, **svr_params)
    sample_weight = make_sample_weights(prepared["y_train_raw_used"], weight_scheme)
    model.fit(prepared["X_train"], prepared["y_train_scaled"], sample_weight=sample_weight)

    pred_train_scaled = model.predict(prepared["X_train"])
    pred_val_scaled = model.predict(prepared["X_val"])

    cal = evaluate_calibrations(
        prepared["target_tf"],
        pred_train_scaled,
        pred_val_scaled,
        prepared["y_train_raw_used"],
        y_val_raw,
    )
    best = cal["best"]
    pred_val_best = best["pred_val_usd"]
    bucket = salary_bucket_summary(y_val_raw, pred_val_best)
    w_summary = sample_weight_summary(prepared["y_train_raw_used"], weight_scheme)

    elapsed = time.time() - start
    record = {
        "feature_profile": feature_profile,
        "target_variant": target_variant,
        "weight_scheme": weight_scheme,
        "drop_low_salary_rows": bool(drop_low_salary_rows),
        "svr_params": json.dumps(svr_params),
        "C": svr_params.get("C"),
        "gamma": str(svr_params.get("gamma")),
        "epsilon": svr_params.get("epsilon"),
        "n_train_rows": prepared["n_train_rows"],
        "n_features": prepared["n_features"],
        "fit_seconds": elapsed,
        "best_calibration": best["calibration"],
        "calibration_info": json.dumps(best.get("factor_or_info", "direct")),
        "val_rmse": float(best["val_rmse"]),
        "val_mae": float(mean_absolute_error(y_val_raw, pred_val_best)),
        "val_pred_mean": float(np.mean(pred_val_best)),
        "val_pred_median": float(np.median(pred_val_best)),
        "val_pred_p95": float(np.quantile(pred_val_best, 0.95)),
        "val_pred_p99": float(np.quantile(pred_val_best, 0.99)),
        "val_pred_max": float(np.max(pred_val_best)),
        "val_high_bucket_rmse_100k_plus": float(rmse(y_val_raw[y_val_raw > HIGH_SALARY_100K], pred_val_best[y_val_raw > HIGH_SALARY_100K])) if (y_val_raw > HIGH_SALARY_100K).sum() > 0 else np.nan,
        "val_high_bucket_rmse_200k_plus": float(rmse(y_val_raw[y_val_raw > HIGH_SALARY_200K], pred_val_best[y_val_raw > HIGH_SALARY_200K])) if (y_val_raw > HIGH_SALARY_200K).sum() > 0 else np.nan,
        "val_low_bucket_rmse_lt10k": float(rmse(y_val_raw[y_val_raw < 10_000], pred_val_best[y_val_raw < 10_000])) if (y_val_raw < 10_000).sum() > 0 else np.nan,
        "weight_max": w_summary["weight_max"],
        "n_weight_gt1": w_summary["n_weight_gt1"],
    }
    if verbose:
        print(record)
        display(bucket)
    return record

print("Weighted candidate evaluation function ready.")


Weighted candidate evaluation function ready.


## 8. Define the weighted tail-focused search grid

In [8]:
# ===============================================================
# 8. Weighted tail-focused RBF SVR grid
# ===============================================================
if FAST_MODE:
    FEATURE_PROFILE_LIST = ["top0_dense", "top5_dense"]
    TARGET_VARIANTS_SEARCH = ["sqrt_raw", "cbrt_raw", "raw_scaled", "log_floor500"]
    WEIGHT_SCHEME_LIST = ["none", "high_mild", "high_medium", "tail_balanced"]
    DROP_LOW_OPTIONS = [False]
    SVR_PARAM_LIST = [
        {"C": 0.1, "gamma": 0.001, "epsilon": 0.1},
        {"C": 0.2, "gamma": 0.001, "epsilon": 0.1},
        {"C": 0.5, "gamma": 0.001, "epsilon": 0.1},
        {"C": 0.2, "gamma": 0.002, "epsilon": 0.1},
        {"C": 0.2, "gamma": 0.0007, "epsilon": 0.05},
    ]
else:
    FEATURE_PROFILE_LIST = ["top0_dense", "top3_dense", "top5_dense", "top5_hybrid_ohe"]
    TARGET_VARIANTS_SEARCH = ["sqrt_raw", "sqrt_floor500", "cbrt_raw", "raw_scaled", "raw_floor500_scaled", "log_floor500"]
    WEIGHT_SCHEME_LIST = ["none", "high_mild", "high_medium", "tail_balanced", "tail_strong"]
    DROP_LOW_OPTIONS = [False, True]
    SVR_PARAM_LIST = list(ParameterGrid({
        "C": [0.05, 0.1, 0.2, 0.35, 0.5, 0.8, 1.0],
        "gamma": [0.0005, 0.0007, 0.001, 0.0015, 0.002, "scale"],
        "epsilon": [0.05, 0.1, 0.2, 0.3],
    }))

n_total = len(FEATURE_PROFILE_LIST) * len(TARGET_VARIANTS_SEARCH) * len(WEIGHT_SCHEME_LIST) * len(DROP_LOW_OPTIONS) * len(SVR_PARAM_LIST)
print("Weighted tail search size:", n_total)
print("Feature profiles:", FEATURE_PROFILE_LIST)
print("Target variants:", TARGET_VARIANTS_SEARCH)
print("Weight schemes:", WEIGHT_SCHEME_LIST)
print("SVR params:", len(SVR_PARAM_LIST))


Weighted tail search size: 160
Feature profiles: ['top0_dense', 'top5_dense']
Target variants: ['sqrt_raw', 'cbrt_raw', 'raw_scaled', 'log_floor500']
Weight schemes: ['none', 'high_mild', 'high_medium', 'tail_balanced']
SVR params: 5


## 9. Run weighted tail search

This cell may take time. Partial results are saved after every candidate.

In [9]:
# ===============================================================
# 9. Run weighted tail search
# ===============================================================
results = []
partial_path = OUTPUT_DIR / "svr_weighted_tail_search_results_partial.csv"
start_all = time.time()

counter = 0
for feature_profile in FEATURE_PROFILE_LIST:
    for target_variant in TARGET_VARIANTS_SEARCH:
        for weight_scheme in WEIGHT_SCHEME_LIST:
            for drop_low in DROP_LOW_OPTIONS:
                for params in SVR_PARAM_LIST:
                    counter += 1
                    print(f"[{counter}/{n_total}] profile={feature_profile}, target={target_variant}, weight={weight_scheme}, drop_low={drop_low}, params={params}")
                    try:
                        rec = evaluate_candidate(
                            feature_profile=feature_profile,
                            target_variant=target_variant,
                            svr_params=params,
                            weight_scheme=weight_scheme,
                            drop_low_salary_rows=drop_low,
                        )
                        results.append(rec)
                        print(f"    val RMSE={rec['val_rmse']:,.0f}, high100 RMSE={rec['val_high_bucket_rmse_100k_plus']:,.0f}, max pred={rec['val_pred_max']:,.0f}, calib={rec['best_calibration']}")
                    except Exception as exc:
                        print("    FAILED:", exc)
                        results.append({
                            "feature_profile": feature_profile,
                            "target_variant": target_variant,
                            "weight_scheme": weight_scheme,
                            "drop_low_salary_rows": bool(drop_low),
                            "svr_params": json.dumps(params),
                            "error": str(exc),
                            "val_rmse": np.nan,
                        })
                    pd.DataFrame(results).to_csv(partial_path, index=False)

results_df = pd.DataFrame(results).sort_values("val_rmse", na_position="last").reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR / "svr_weighted_tail_search_results.csv", index=False)

print("Search finished in minutes:", (time.time() - start_all) / 60)
display(results_df.head(30)[[
    "feature_profile", "target_variant", "weight_scheme", "drop_low_salary_rows", "C", "gamma", "epsilon",
    "best_calibration", "val_rmse", "val_pred_max", "val_high_bucket_rmse_100k_plus", "val_high_bucket_rmse_200k_plus",
    "val_low_bucket_rmse_lt10k", "n_features", "fit_seconds"
]])


[1/160] profile=top0_dense, target=sqrt_raw, weight=none, drop_low=False, params={'C': 0.1, 'gamma': 0.001, 'epsilon': 0.1}
    val RMSE=37,602, high100 RMSE=91,882, max pred=120,650, calib=train_binned_ratio
[2/160] profile=top0_dense, target=sqrt_raw, weight=none, drop_low=False, params={'C': 0.2, 'gamma': 0.001, 'epsilon': 0.1}
    val RMSE=37,664, high100 RMSE=93,535, max pred=117,344, calib=train_global_ratio
[3/160] profile=top0_dense, target=sqrt_raw, weight=none, drop_low=False, params={'C': 0.5, 'gamma': 0.001, 'epsilon': 0.1}
    val RMSE=37,859, high100 RMSE=92,609, max pred=129,119, calib=train_global_ratio
[4/160] profile=top0_dense, target=sqrt_raw, weight=none, drop_low=False, params={'C': 0.2, 'gamma': 0.002, 'epsilon': 0.1}
    val RMSE=37,939, high100 RMSE=94,824, max pred=113,268, calib=train_global_ratio
[5/160] profile=top0_dense, target=sqrt_raw, weight=none, drop_low=False, params={'C': 0.2, 'gamma': 0.0007, 'epsilon': 0.05}
    val RMSE=37,550, high100 RMSE=93,2

,feature_profile,target_variant,weight_scheme,drop_low_salary_rows,C,gamma,epsilon,best_calibration,val_rmse,val_pred_max,val_high_bucket_rmse_100k_plus,val_high_bucket_rmse_200k_plus,val_low_bucket_rmse_lt10k,n_features,fit_seconds
0,top5_dense,log_floor500,high_mild,False,0.2,0.0007,0.05,none,36900.793838,158107.574748,90264.246589,251210.168913,39796.489205,376,11.253115
1,top0_dense,log_floor500,high_mild,False,0.2,0.0007,0.05,none,36933.754154,154787.888486,91099.152604,253424.711859,38884.128698,301,10.429486
2,top5_dense,log_floor500,high_mild,False,0.1,0.001,0.10,none,37025.058344,144042.580626,92036.610911,255735.733286,39073.690226,376,10.645571
3,top5_dense,cbrt_raw,high_mild,False,0.2,0.0007,0.05,none,37028.176309,134779.086538,90510.492157,254812.064055,40489.372350,376,11.389816
4,top0_dense,cbrt_raw,high_mild,False,0.2,0.0007,0.05,none,37107.329597,133957.845710,91358.809277,255880.075532,39731.947506,301,10.541835
5,top5_dense,log_floor500,high_medium,False,0.2,0.0007,0.05,train_binned_ratio,37120.688986,165083.813945,87055.620703,244545.912042,43242.284929,376,11.061921
6,top0_dense,log_floor500,high_mild,False,0.1,0.001,0.10,none,37155.384653,145004.922922,92388.664976,256174.223987,38713.874117,301,10.739855
7,top5_dense,cbrt_raw,high_mild,False,0.2,0.001,0.10,none,37159.122683,132981.155926,90986.711965,255419.982967,40657.062578,376,10.989876
8,top0_dense,log_floor500,high_medium,False,0.2,0.0007,0.05,none,37168.800081,171777.979900,87954.180710,245433.620931,41707.407423,301,10.213691
9,top5_dense,log_floor500,high_medium,False,0.1,0.001,0.10,none,37170.602892,156671.096856,88255.476740,248586.716929,42459.536728,376,10.756558


## 10. Fold-safe CV for top candidates

This fixes the previous problem where the automatically selected model had no CV values.

In [10]:
# ===============================================================
# 10. Fold-safe CV for top weighted candidates
# ===============================================================
def make_cv_splitter_for_train(train_df: pd.DataFrame, n_splits: int = N_SPLITS_CV):
    if "region" in train_df.columns:
        strata = make_region_strata(train_df["region"], min_count=n_splits)
        counts = strata.value_counts()
        if len(counts) >= 2 and counts.min() >= n_splits:
            return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE), strata
    return KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE), None


def fold_safe_cv_candidate(row: pd.Series) -> Dict[str, Any]:
    params = json.loads(row["svr_params"])
    feature_profile = row["feature_profile"]
    target_variant = row["target_variant"]
    weight_scheme = row["weight_scheme"]
    drop_low = bool(row["drop_low_salary_rows"])

    splitter, strata = make_cv_splitter_for_train(df_train)
    split_iter = splitter.split(df_train, strata) if strata is not None else splitter.split(df_train)
    fold_rmse = []
    fold_mae = []
    fold_high_rmse = []
    fold_max_pred = []

    for fold_id, (tr_idx, va_idx) in enumerate(split_iter, start=1):
        fold_train = df_train.iloc[tr_idx].reset_index(drop=True)
        fold_val = df_train.iloc[va_idx].reset_index(drop=True)
        y_fold_val_raw = fold_val[TARGET].astype(float).reset_index(drop=True)

        prepared = prepare_candidate_data(
            feature_profile=feature_profile,
            target_variant=target_variant,
            drop_low_salary_rows=drop_low,
            train_df=fold_train,
            val_df=fold_val,
            kaggle_df=None,
        )
        model = SVR(kernel="rbf", cache_size=1000, **params)
        sample_weight = make_sample_weights(prepared["y_train_raw_used"], weight_scheme)
        model.fit(prepared["X_train"], prepared["y_train_scaled"], sample_weight=sample_weight)

        pred_train_scaled = model.predict(prepared["X_train"])
        pred_val_scaled = model.predict(prepared["X_val"])
        cal = evaluate_calibrations(
            prepared["target_tf"],
            pred_train_scaled,
            pred_val_scaled,
            prepared["y_train_raw_used"],
            y_fold_val_raw,
        )
        pred_val_usd = cal["best"]["pred_val_usd"]
        fold_rmse.append(rmse(y_fold_val_raw, pred_val_usd))
        fold_mae.append(mean_absolute_error(y_fold_val_raw, pred_val_usd))
        if (y_fold_val_raw > HIGH_SALARY_100K).sum() > 0:
            fold_high_rmse.append(rmse(y_fold_val_raw[y_fold_val_raw > HIGH_SALARY_100K], pred_val_usd[y_fold_val_raw > HIGH_SALARY_100K]))
        fold_max_pred.append(float(np.max(pred_val_usd)))

    return {
        "feature_profile": feature_profile,
        "target_variant": target_variant,
        "weight_scheme": weight_scheme,
        "drop_low_salary_rows": drop_low,
        "svr_params": row["svr_params"],
        "cv_rmse_mean": float(np.mean(fold_rmse)),
        "cv_rmse_median": float(np.median(fold_rmse)),
        "cv_rmse_std": float(np.std(fold_rmse, ddof=1)),
        "cv_rmse_min": float(np.min(fold_rmse)),
        "cv_rmse_max": float(np.max(fold_rmse)),
        "cv_mae_mean": float(np.mean(fold_mae)),
        "cv_high100_rmse_mean": float(np.mean(fold_high_rmse)) if len(fold_high_rmse) else np.nan,
        "cv_max_pred_mean": float(np.mean(fold_max_pred)),
        "fold_scores": json.dumps([float(x) for x in fold_rmse]),
    }

# Run CV for top validation candidates and also top high-bucket candidates.
TOP_N_FOR_CV = 12 if FAST_MODE else 30
cv_by_val = results_df.dropna(subset=["val_rmse"]).head(TOP_N_FOR_CV)
cv_by_high = results_df.dropna(subset=["val_high_bucket_rmse_100k_plus"]).sort_values("val_high_bucket_rmse_100k_plus").head(max(4, TOP_N_FOR_CV // 3))
cv_input = pd.concat([cv_by_val, cv_by_high], ignore_index=True).drop_duplicates(
    subset=["feature_profile", "target_variant", "weight_scheme", "drop_low_salary_rows", "svr_params"]
).reset_index(drop=True)

cv_records = []
for i, row in cv_input.iterrows():
    print(f"CV [{i+1}/{len(cv_input)}] {row['feature_profile']} | {row['target_variant']} | weight={row['weight_scheme']} | {row['svr_params']}")
    rec = fold_safe_cv_candidate(row)
    cv_records.append(rec)
    print(f"    CV mean={rec['cv_rmse_mean']:,.0f}, median={rec['cv_rmse_median']:,.0f}, std={rec['cv_rmse_std']:,.0f}, max={rec['cv_rmse_max']:,.0f}")

cv_df = pd.DataFrame(cv_records).sort_values("cv_rmse_median")
cv_df.to_csv(OUTPUT_DIR / "svr_weighted_tail_top_candidates_fold_safe_cv.csv", index=False)
print("Fold-safe CV results:")
display(cv_df)


CV [1/15] top5_dense | log_floor500 | weight=high_mild | {"C": 0.2, "gamma": 0.0007, "epsilon": 0.05}
    CV mean=96,099, median=60,966, std=88,784, max=253,456
CV [2/15] top0_dense | log_floor500 | weight=high_mild | {"C": 0.2, "gamma": 0.0007, "epsilon": 0.05}
    CV mean=96,137, median=60,945, std=88,720, max=253,382
CV [3/15] top5_dense | log_floor500 | weight=high_mild | {"C": 0.1, "gamma": 0.001, "epsilon": 0.1}
    CV mean=96,026, median=60,967, std=88,768, max=253,398
CV [4/15] top5_dense | cbrt_raw | weight=high_mild | {"C": 0.2, "gamma": 0.0007, "epsilon": 0.05}
    CV mean=95,921, median=60,852, std=88,721, max=253,189
CV [5/15] top0_dense | cbrt_raw | weight=high_mild | {"C": 0.2, "gamma": 0.0007, "epsilon": 0.05}
    CV mean=95,982, median=60,803, std=88,764, max=253,331
CV [6/15] top5_dense | log_floor500 | weight=high_medium | {"C": 0.2, "gamma": 0.0007, "epsilon": 0.05}
    CV mean=96,413, median=61,042, std=88,659, max=253,548
CV [7/15] top0_dense | log_floor500 | weig

,feature_profile,target_variant,weight_scheme,drop_low_salary_rows,svr_params,cv_rmse_mean,cv_rmse_median,cv_rmse_std,cv_rmse_min,cv_rmse_max,cv_mae_mean,cv_high100_rmse_mean,cv_max_pred_mean,fold_scores
4,top0_dense,cbrt_raw,high_mild,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",95982.118821,60803.217380,88763.914233,44870.700786,253331.459616,27782.000090,312613.090822,130185.784696,"[253331.45961638525, 44870.70078601984, 46619...."
3,top5_dense,cbrt_raw,high_mild,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",95921.219594,60852.394923,88720.991945,44904.256323,253189.349087,27675.006513,312354.159888,129681.570805,"[253189.34908711317, 44904.25632306006, 46430...."
12,top5_dense,cbrt_raw,high_medium,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",96259.952057,60874.293283,88646.271053,45392.582071,253394.027901,28356.669414,311508.153408,140905.869690,"[253394.02790124575, 45392.58207081175, 46858...."
13,top5_dense,sqrt_raw,high_medium,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",96235.085650,60882.382161,88632.030285,45385.072811,253353.532696,28383.019112,311439.210804,135361.382582,"[253353.53269566226, 45385.07281057698, 46879...."
1,top0_dense,log_floor500,high_mild,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",96136.643373,60944.575802,88719.927220,45006.401512,253381.516672,27943.367727,312500.764445,144944.686494,"[253381.51667183437, 45006.401511630385, 46690..."
0,top5_dense,log_floor500,high_mild,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",96099.149212,60965.686095,88783.856208,44900.749528,253456.402099,27792.557787,312734.535971,145472.650464,"[253456.4020991437, 44900.749527614666, 46603...."
2,top5_dense,log_floor500,high_mild,False,"{""C"": 0.1, ""gamma"": 0.001, ""epsilon"": 0.1}",96025.609436,60967.043532,88768.035768,44899.205082,253398.349458,27625.854415,313183.758862,131109.645123,"[253398.34945835307, 44899.20508242198, 46720...."
7,top5_dense,cbrt_raw,high_mild,False,"{""C"": 0.2, ""gamma"": 0.001, ""epsilon"": 0.1}",95989.661647,60971.089401,88739.586701,44918.948040,253297.167072,27617.935706,312956.640674,126547.723367,"[253297.1670718984, 44918.948039831324, 46548...."
9,top5_dense,log_floor500,high_medium,False,"{""C"": 0.1, ""gamma"": 0.001, ""epsilon"": 0.1}",96233.042061,60984.830201,88761.770465,44968.411016,253565.031977,28225.807663,312321.396176,145090.578344,"[253565.03197654308, 44968.411016418584, 46983..."
6,top0_dense,log_floor500,high_mild,False,"{""C"": 0.1, ""gamma"": 0.001, ""epsilon"": 0.1}",96066.892383,61007.161687,88760.485207,44978.913291,253428.311680,27896.216012,312664.285497,134177.163917,"[253428.3116801963, 44978.91329084143, 46744.7..."


## 11. Select a CV-checked candidate

In [11]:
# ===============================================================
# 11. Candidate selection — only among CV-evaluated candidates
# ===============================================================
key_cols = ["feature_profile", "target_variant", "weight_scheme", "drop_low_salary_rows", "svr_params"]
merged = results_df.merge(cv_df, on=key_cols, how="left")
merged["has_cv"] = merged["cv_rmse_mean"].notna()

# Important correction vs previous notebook: final selection is made only among candidates with fold-safe CV.
selection_pool = merged[merged["has_cv"]].copy()
if selection_pool.empty:
    print("WARNING: no CV-evaluated candidates found; falling back to validation ranking.")
    selection_pool = merged.copy()

# Combined score: validation RMSE remains important, but unstable CV is penalized.
# cv_rmse_mean can be dominated by a single extreme fold, so cv_median is also included.
selection_pool["selection_score"] = (
    selection_pool["val_rmse"]
    + 0.20 * np.maximum(selection_pool["cv_rmse_median"].fillna(selection_pool["val_rmse"]) - selection_pool["val_rmse"], 0)
    + 0.03 * selection_pool["cv_rmse_std"].fillna(0)
)

selection_table = selection_pool.sort_values("selection_score").head(30).reset_index(drop=True)
selection_table.to_csv(OUTPUT_DIR / "svr_weighted_tail_selection_table.csv", index=False)
display(selection_table[[
    "feature_profile", "target_variant", "weight_scheme", "drop_low_salary_rows", "svr_params", "best_calibration",
    "val_rmse", "cv_rmse_mean", "cv_rmse_median", "cv_rmse_std", "val_pred_max", "val_high_bucket_rmse_100k_plus", "selection_score"
]])

selected_row = selection_table.iloc[0].to_dict()
print("Selected CV-checked candidate:")
for k in ["feature_profile", "target_variant", "weight_scheme", "drop_low_salary_rows", "svr_params", "val_rmse", "best_calibration", "n_features", "cv_rmse_mean", "cv_rmse_median", "cv_rmse_std", "selection_score"]:
    print(f"{k}: {selected_row.get(k)}")

with open(OUTPUT_DIR / "selected_weighted_tail_candidate.json", "w", encoding="utf-8") as f:
    json.dump(selected_row, f, indent=2)


,feature_profile,target_variant,weight_scheme,drop_low_salary_rows,svr_params,best_calibration,val_rmse,cv_rmse_mean,cv_rmse_median,cv_rmse_std,val_pred_max,val_high_bucket_rmse_100k_plus,selection_score
0,top5_dense,log_floor500,high_mild,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",none,36900.793838,96099.149212,60965.686095,88783.856208,158107.574748,90264.246589,44377.287975
1,top0_dense,log_floor500,high_mild,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",none,36933.754154,96136.643373,60944.575802,88719.927220,154787.888486,91099.152604,44397.516301
2,top5_dense,cbrt_raw,high_mild,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",none,37028.176309,95921.219594,60852.394923,88720.991945,134779.086538,90510.492157,44454.649790
3,top5_dense,log_floor500,high_mild,False,"{""C"": 0.1, ""gamma"": 0.001, ""epsilon"": 0.1}",none,37025.058344,96025.609436,60967.043532,88768.035768,144042.580626,92036.610911,44476.496454
4,top0_dense,cbrt_raw,high_mild,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",none,37107.329597,95982.118821,60803.217380,88763.914233,133957.845710,91358.809277,44509.424581
5,top5_dense,log_floor500,high_medium,False,"{""C"": 0.2, ""gamma"": 0.0007, ""epsilon"": 0.05}",train_binned_ratio,37120.688986,96412.791747,61041.910128,88659.122380,165083.813945,87055.620703,44564.706886
6,top5_dense,cbrt_raw,high_mild,False,"{""C"": 0.2, ""gamma"": 0.001, ""epsilon"": 0.1}",none,37159.122683,95989.661647,60971.089401,88739.586701,132981.155926,90986.711965,44583.703628
7,top0_dense,log_floor500,high_mild,False,"{""C"": 0.1, ""gamma"": 0.001, ""epsilon"": 0.1}",none,37155.384653,96066.892383,61007.161687,88760.485207,145004.922922,92388.664976,44588.554616
8,top5_dense,log_floor500,high_medium,False,"{""C"": 0.1, ""gamma"": 0.001, ""epsilon"": 0.1}",none,37170.602892,96233.042061,60984.830201,88761.770465,156671.096856,88255.476740,44596.301468
9,top0_dense,log_floor500,high_medium,False,"{""C"": 0.1, ""gamma"": 0.001, ""epsilon"": 0.1}",none,37191.924840,96335.884346,61019.734291,88709.592111,158320.548945,89233.760039,44618.774494


Selected CV-checked candidate:
feature_profile: top5_dense
target_variant: log_floor500
weight_scheme: high_mild
drop_low_salary_rows: False
svr_params: {"C": 0.2, "gamma": 0.0007, "epsilon": 0.05}
val_rmse: 36900.793837824145
best_calibration: none
n_features: 376
cv_rmse_mean: 96099.1492121258
cv_rmse_median: 60965.68609483198
cv_rmse_std: 88783.85620780077
selection_score: 44377.28797545974


## 12. Locked internal-test evaluation

In [12]:
# ===============================================================
# 12. Internal-test evaluation for selected weighted candidate
# ===============================================================
def fit_candidate_and_predict(
    selected: Dict[str, Any],
    train_df_fit: pd.DataFrame,
    predict_df: pd.DataFrame,
    kaggle_df_optional: Optional[pd.DataFrame] = None,
):
    params = json.loads(selected["svr_params"])
    profile = selected["feature_profile"]
    target_variant = selected["target_variant"]
    drop_low = bool(selected.get("drop_low_salary_rows", False))
    weight_scheme = selected.get("weight_scheme", "none")

    prepared = prepare_candidate_data(
        feature_profile=profile,
        target_variant=target_variant,
        drop_low_salary_rows=drop_low,
        train_df=train_df_fit,
        val_df=predict_df,
        kaggle_df=kaggle_df_optional,
    )
    model = SVR(kernel="rbf", cache_size=1000, **params)
    sample_weight = make_sample_weights(prepared["y_train_raw_used"], weight_scheme)
    model.fit(prepared["X_train"], prepared["y_train_scaled"], sample_weight=sample_weight)

    pred_train_scaled = model.predict(prepared["X_train"])
    pred_eval_scaled = model.predict(prepared["X_val"])
    pred_eval_usd_none = prepared["target_tf"].inverse_transform_to_usd(pred_eval_scaled)

    # Refit selected train-only calibration on the current training data.
    calibration = selected.get("best_calibration", "none")
    pred_train_usd = prepared["target_tf"].inverse_transform_to_usd(pred_train_scaled)
    if calibration == "train_global_ratio":
        factor = fit_train_global_ratio(pred_train_usd, prepared["y_train_raw_used"])
        pred_eval_usd = np.clip(pred_eval_usd_none * factor, PRED_MIN_USD, PRED_MAX_USD)
        calib_info = {"calibration": calibration, "factor": factor}
    elif calibration == "train_binned_ratio":
        binned = BinnedRatioCalibrator(n_bins=6).fit(pred_train_usd, prepared["y_train_raw_used"])
        pred_eval_usd = binned.apply(pred_eval_usd_none)
        calib_info = {"calibration": calibration, "global_factor": binned.global_factor_, "bin_factors": binned.bin_factors_.tolist()}
    else:
        pred_eval_usd = pred_eval_usd_none
        calib_info = {"calibration": "none"}

    pred_kaggle_usd = None
    if prepared["X_kaggle"] is not None:
        pred_kaggle_scaled = model.predict(prepared["X_kaggle"])
        pred_kaggle_usd_none = prepared["target_tf"].inverse_transform_to_usd(pred_kaggle_scaled)
        if calibration == "train_global_ratio":
            pred_kaggle_usd = np.clip(pred_kaggle_usd_none * calib_info["factor"], PRED_MIN_USD, PRED_MAX_USD)
        elif calibration == "train_binned_ratio":
            pred_kaggle_usd = binned.apply(pred_kaggle_usd_none)
        else:
            pred_kaggle_usd = pred_kaggle_usd_none

    return pred_eval_usd, pred_kaggle_usd, prepared, model, calib_info


pred_internal_usd, _, prepared_selected, selected_model, calib_info = fit_candidate_and_predict(
    selected_row,
    train_df_fit=df_train,
    predict_df=df_internal_test,
    kaggle_df_optional=None,
)

internal_rmse = rmse(y_internal_test_raw, pred_internal_usd)
internal_mae = mean_absolute_error(y_internal_test_raw, pred_internal_usd)
print("Internal-test RMSE:", f"${internal_rmse:,.0f}")
print("Internal-test MAE:", f"${internal_mae:,.0f}")
print("Calibration info:", calib_info)

internal_bucket = salary_bucket_summary(y_internal_test_raw, pred_internal_usd)
display(internal_bucket)

internal_pred_df = pd.DataFrame({
    "actual_salary": y_internal_test_raw,
    "predicted_salary": pred_internal_usd,
    "error": pred_internal_usd - y_internal_test_raw,
    "abs_error": np.abs(pred_internal_usd - y_internal_test_raw),
})
internal_pred_df.to_csv(OUTPUT_DIR / "selected_weighted_candidate_internal_test_predictions.csv", index=False)
internal_bucket.to_csv(OUTPUT_DIR / "selected_weighted_candidate_internal_test_bucket_summary.csv", index=False)

internal_summary = {
    "internal_test_rmse": internal_rmse,
    "internal_test_mae": float(internal_mae),
    "selected_candidate": selected_row,
    "calibration_info": calib_info,
}
with open(OUTPUT_DIR / "selected_weighted_candidate_internal_test_summary.json", "w", encoding="utf-8") as f:
    json.dump(internal_summary, f, indent=2)


Internal-test RMSE: $50,674
Internal-test MAE: $23,467
Calibration info: {'calibration': 'none'}


,bucket,n,mean_actual,mean_pred,rmse,mae,mse_sum,mse_contribution_pct
0,<500,12,249.250000,30008.013776,33888.504606,29758.763776,1.378117e+10,1.423577
1,500-10k,58,2718.844828,39567.289721,42258.545741,36848.444894,1.035755e+11,10.699220
2,10k-30k,73,21180.684932,28452.447477,15820.922149,10817.690505,1.827202e+10,1.887476
3,30k-60k,137,45077.379562,48667.809781,20983.294847,15773.677328,6.032092e+10,6.231075
4,60k-100k,74,74173.378378,62417.759194,23644.399608,19060.566476,4.137026e+10,4.273496
5,100k-200k,21,132373.000000,83024.363189,58876.019017,53164.674092,7.279410e+10,7.519538
6,200k+,2,557293.000000,119794.725709,573564.323683,437498.274291,6.579521e+11,67.965618


## 13. Generate selected and backup submissions

In [13]:
# ===============================================================
# 13. Generate multiple Kaggle submissions
# ===============================================================
def make_submission_for_candidate(candidate: Dict[str, Any], train_df_fit: pd.DataFrame, filename: str) -> pd.DataFrame:
    _, pred_kaggle, _, _, _ = fit_candidate_and_predict(
        candidate,
        train_df_fit=train_df_fit,
        predict_df=df_val,  # placeholder required for matrix construction
        kaggle_df_optional=df_kaggle,
    )
    sub = pd.DataFrame({
        "id": df_kaggle[ID_COL].astype(int),
        "annual.pay.usd": pred_kaggle.astype(float),
    })
    assert sub.shape[0] == len(df_kaggle)
    assert sub["id"].is_unique
    assert sub["annual.pay.usd"].notna().all()
    assert (sub["annual.pay.usd"] > 0).all()
    sub.to_csv(OUTPUT_DIR / filename, index=False)
    return sub

# Selected candidate: train-only and train+validation refit.
submission_selected_train_only = make_submission_for_candidate(
    selected_row,
    train_df_fit=df_train,
    filename="submission_weighted_tail_selected_train_only.csv",
)

df_train_plus_val = pd.concat([df_train, df_val], axis=0).reset_index(drop=True)
submission_selected_refit = make_submission_for_candidate(
    selected_row,
    train_df_fit=df_train_plus_val,
    filename="submission_weighted_tail_selected_refit_train_val.csv",
)

# Also create backup submissions for the next two CV-checked candidates.
backup_summaries = []
for i in range(1, min(3, len(selection_table))):
    cand = selection_table.iloc[i].to_dict()
    fname = f"submission_weighted_tail_backup_{i}_refit_train_val.csv"
    sub = make_submission_for_candidate(cand, train_df_fit=df_train_plus_val, filename=fname)
    backup_summaries.append({
        "rank": i + 1,
        "filename": fname,
        "feature_profile": cand.get("feature_profile"),
        "target_variant": cand.get("target_variant"),
        "weight_scheme": cand.get("weight_scheme"),
        "val_rmse": cand.get("val_rmse"),
        "cv_rmse_median": cand.get("cv_rmse_median"),
        "prediction_mean": float(sub["annual.pay.usd"].mean()),
        "prediction_p95": float(sub["annual.pay.usd"].quantile(0.95)),
        "prediction_max": float(sub["annual.pay.usd"].max()),
    })

print("Selected train-only prediction distribution:")
display(submission_selected_train_only["annual.pay.usd"].describe().to_frame())
print("Selected refit train+validation prediction distribution:")
display(submission_selected_refit["annual.pay.usd"].describe().to_frame())
if backup_summaries:
    backup_df = pd.DataFrame(backup_summaries)
    backup_df.to_csv(OUTPUT_DIR / "backup_submission_summary.csv", index=False)
    display(backup_df)

print("Saved selected submissions:")
print((OUTPUT_DIR / "submission_weighted_tail_selected_train_only.csv").resolve())
print((OUTPUT_DIR / "submission_weighted_tail_selected_refit_train_val.csv").resolve())


Selected train-only prediction distribution:


,annual.pay.usd
count,628.000000
mean,46887.964279
std,26342.170620
min,7325.355624
25%,27000.575877
50%,41238.230142
75%,62076.381515
max,172484.880713


Selected refit train+validation prediction distribution:


,annual.pay.usd
count,628.000000
mean,46432.519670
std,26036.556883
min,6572.868614
25%,26867.089849
50%,41491.020990
75%,60776.345719
max,174161.176315


,rank,filename,feature_profile,target_variant,weight_scheme,val_rmse,cv_rmse_median,prediction_mean,prediction_p95,prediction_max
0,2,submission_weighted_tail_backup_1_refit_train_...,top0_dense,log_floor500,high_mild,36933.754154,60944.575802,46347.913073,95302.026773,174912.383076
1,3,submission_weighted_tail_backup_2_refit_train_...,top5_dense,cbrt_raw,high_mild,37028.176309,60852.394923,47063.813105,93196.490658,151349.408044


Saved selected submissions:
C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\svr_tail_weighted_outputs\submission_weighted_tail_selected_train_only.csv
C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\svr_tail_weighted_outputs\submission_weighted_tail_selected_refit_train_val.csv


## 14. Save final summary

In [14]:
# ===============================================================
# 14. Save final weighted-tail summary
# ===============================================================
summary = {
    "notebook": "06_svr_rbf_weighted_tail_search.ipynb",
    "algorithm": "SVR with RBF kernel only",
    "fast_mode": FAST_MODE,
    "purpose": "Address high-salary underprediction using sample weights, dense tail-risk features, alternative targets, and CV-checked selection.",
    "split_summary": split_summary.to_dict(orient="records"),
    "n_candidates_evaluated": int(len(results_df)),
    "best_validation_candidate": results_df.iloc[0].to_dict() if len(results_df) else None,
    "selected_candidate": selected_row,
    "internal_test": {
        "rmse": internal_rmse,
        "mae": float(internal_mae),
    },
    "created_files": {
        "search_results": str((OUTPUT_DIR / "svr_weighted_tail_search_results.csv").resolve()),
        "cv_results": str((OUTPUT_DIR / "svr_weighted_tail_top_candidates_fold_safe_cv.csv").resolve()),
        "selection_table": str((OUTPUT_DIR / "svr_weighted_tail_selection_table.csv").resolve()),
        "submission_selected_train_only": str((OUTPUT_DIR / "submission_weighted_tail_selected_train_only.csv").resolve()),
        "submission_selected_refit_train_val": str((OUTPUT_DIR / "submission_weighted_tail_selected_refit_train_val.csv").resolve()),
    },
    "method_notes": [
        "No validation multiplier is used.",
        "Sample weights are normalized to mean 1 so C remains comparable.",
        "Target encodings and scalers are refitted inside fold-safe CV.",
        "Final selection is made only among CV-evaluated candidates.",
        "Multiple submissions are generated because public leaderboard performance can be affected by rare extreme salaries.",
    ],
}
with open(OUTPUT_DIR / "svr_weighted_tail_final_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to:", (OUTPUT_DIR / "svr_weighted_tail_final_summary.json").resolve())
print(json.dumps(summary, indent=2)[:3500])


Summary saved to: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\svr_tail_weighted_outputs\svr_weighted_tail_final_summary.json
{
  "notebook": "06_svr_rbf_weighted_tail_search.ipynb",
  "algorithm": "SVR with RBF kernel only",
  "fast_mode": true,
  "purpose": "Address high-salary underprediction using sample weights, dense tail-risk features, alternative targets, and CV-checked selection.",
  "split_summary": [
    {
      "split": "train",
      "n": 1758,
      "mean": 51287.04607508532,
      "median": 41854.5,
      "min": 11.0,
      "max": 4773360.0,
      "lt_500": 57,
      "gt_100k": 143,
      "gt_200k": 13,
      "p01": 76.26,
      "p05": 882.5500000000001,
      "p25": 16073.5,
      "p75": 67898.0,
      "p95": 117887.44999999998,
      "p99": 178901.9000000001
    },
    {
      "split": "validation",
      "n": 377,
      "mean": 46288.106100795754,
      "median"

# How to use this notebook

Start with `FAST_MODE=True`. Inspect:

```text
svr_tail_weighted_outputs/svr_weighted_tail_search_results.csv
svr_tail_weighted_outputs/svr_weighted_tail_top_candidates_fold_safe_cv.csv
svr_tail_weighted_outputs/svr_weighted_tail_selection_table.csv
svr_tail_weighted_outputs/selected_weighted_candidate_internal_test_bucket_summary.csv
```

Then compare the generated submissions:

```text
submission_weighted_tail_selected_train_only.csv
submission_weighted_tail_selected_refit_train_val.csv
submission_weighted_tail_backup_1_refit_train_val.csv
submission_weighted_tail_backup_2_refit_train_val.csv
```

The most important diagnostic is not only total RMSE, but also whether the model increases the upper-tail prediction range without destroying the middle-salary buckets.
